# MATH 616: Computational Linear Algebra
## Coding Lab — Matrix Factorization

This notebook is the coding companion to the *Matrix Factorization* lecture. We implement each algorithm from the slides in Python, check it against the worked examples, and then run experiments on the questions the slides raise: why factoring pays off, what structure buys you, why the permutation matters, and how conditioning differs from stability.

### Learning goals
By the end of the lab you should be able to:

- build the Gauss transformations $M_k$ and see $L$ appear as the stored multipliers;
- implement LU (elimination, Doolittle and Crout), and use one factorization to solve many right-hand sides;
- test the leading-principal-minor condition, and see what happens when it fails;
- implement Cholesky and the tridiagonal (Thomas) factorization, and measure what they save;
- implement $PA = LU$ with partial pivoting, including the row swap of $L$ that is easy to forget;
- compute condition numbers, see why a small residual does not mean a small error, and recover digits with iterative refinement;
- separate an *ill-conditioned problem* from an *unstable algorithm* using backward error and the growth factor.

Sections marked ✏️ are for you to edit and re-run.

## 0. Imports and helpers

In [ ]:
import time
import numpy as np
import scipy.linalg as sla
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True, linewidth=110)
rng = np.random.default_rng(616)


def forward(L, b):
    # Solve L z = b for a lower-triangular L (b may be 1-D or 2-D).
    b = np.asarray(b, dtype=float)
    z = np.zeros_like(b)
    for i in range(L.shape[0]):
        z[i] = (b[i] - L[i, :i] @ z[:i]) / L[i, i]
    return z


def backward(U, z):
    # Solve U x = z for an upper-triangular U (z may be 1-D or 2-D).
    z = np.asarray(z, dtype=float)
    x = np.zeros_like(z)
    for i in range(U.shape[0] - 1, -1, -1):
        x[i] = (z[i] - U[i, i+1:] @ x[i+1:]) / U[i, i]
    return x


def rel_err(x, x_true):
    return np.linalg.norm(x - x_true) / np.linalg.norm(x_true)


def time_it(f, *args, repeats=3):
    # Best wall-clock time of f(*args) over a few repeats, in seconds.
    best = np.inf
    for _ in range(repeats):
        t0 = time.perf_counter()
        f(*args)
        best = min(best, time.perf_counter() - t0)
    return best

# Part I. The LU factorization

## 1. Gaussian elimination as matrix multiplication

Step $k$ of elimination multiplies by the Gauss transformation

$$M_k = I - \tau^{(k)} e_k^T, \qquad \tau^{(k)}_j = \begin{cases} 0 & j \le k \\ a^{(k-1)}_{jk}/a^{(k-1)}_{kk} & j > k.\end{cases}$$

(Python counts from 0, so "step $k$" on the slides is `k-1` in the code.) We build each $M_k$ as an explicit matrix and apply it to the $4\times 4$ example from the slides.

In [ ]:
def gauss_transform(A, k):
    # The Gauss transformation M_k that zeros column k of A below the diagonal.
    n = A.shape[0]
    tau = np.zeros(n)
    tau[k+1:] = A[k+1:, k] / A[k, k]
    e_k = np.zeros(n)
    e_k[k] = 1.0
    return np.eye(n) - np.outer(tau, e_k), tau


A = np.array([[ 2.,  1.,  2.,  0.],
              [ 4.,  1.,  6.,  2.],
              [-6., -1., -7., -3.],
              [ 8.,  3., 16., 10.]])

Ak = A.copy()
Ms, taus = [], []
for k in range(3):
    M, tau = gauss_transform(Ak, k)
    Ak = M @ Ak
    Ms.append(M)
    taus.append(tau)
    print(f"step {k+1}: multipliers tau = {tau}")
    print(Ak, "\n")
U = Ak

**Check the slide's claims.** (a) $M_k^{-1} = I + \tau^{(k)} e_k^T$: just flip the signs. (b) The product $M_1^{-1} M_2^{-1} M_3^{-1}$ needs no arithmetic at all: it is the identity with the multipliers written below the diagonal.

In [ ]:
n = A.shape[0]
Minv = [np.eye(n) + (np.eye(n) - M) for M in Ms]      # I + tau e_k^T
for k, (M, Mi) in enumerate(zip(Ms, Minv), start=1):
    print(f"M_{k} @ (I + tau e_{k}^T) == I :", np.allclose(M @ Mi, np.eye(n)))

L = Minv[0] @ Minv[1] @ Minv[2]
print("\nL = M_1^-1 M_2^-1 M_3^-1 =\n", L)
print("\nL @ U == A :", np.allclose(L @ U, A))

Compare with the slides: $L$ holds the multipliers $2, -3, 4$ in column 1, then $-2, 1$, then $2$.

## 2. The LU algorithm from the slides

Forming each $M_k$ as an $n\times n$ matrix wastes $O(n^3)$ work per step. The algorithm writes the multipliers straight into $L$ and updates the trailing block with one outer product.

In [ ]:
def mylu(A):
    # Factor A = L @ U by elimination, no row interchanges.
    A = A.astype(float).copy()
    n = A.shape[0]
    L, U = np.eye(n), np.zeros((n, n))
    U[0] = A[0]
    for i in range(n-1):
        L[i+1:, i] = A[i+1:, i] / U[i, i]
        A[i+1:, i:] -= np.outer(L[i+1:, i], U[i, i:])
        U[i+1, i+1:] = A[i+1, i+1:]
    return L, U


L, U = mylu(A)
print("L =\n", L)
print("U =\n", U)
print("||L U - A|| =", np.linalg.norm(L @ U - A))

### ✏️ Your turn
`mylu` never copies `U[i+1, i+1]`, the diagonal entry, into `U` explicitly. Yet the diagonal of `U` comes out right. Find the line that puts it there. (Hint: which slice does `U[i+1, i+1:]` start at?)

## 3. Doolittle and Crout

Both compute $L$ and $U$ **directly from the entries of the product** $LU = A$, one row and column at a time. Doolittle fixes $\ell_{ii} = 1$; Crout fixes $u_{ii} = 1$.

In [ ]:
def doolittle(A):
    # A = L U with unit lower-triangular L (l_ii = 1).
    n = A.shape[0]
    L, U = np.eye(n), np.zeros((n, n))
    for k in range(n):
        U[k, k:] = A[k, k:] - L[k, :k] @ U[:k, k:]                     # row k of U
        L[k+1:, k] = (A[k+1:, k] - L[k+1:, :k] @ U[:k, k]) / U[k, k]   # column k of L
    return L, U


def crout(A):
    # A = L U with unit upper-triangular U (u_ii = 1).
    n = A.shape[0]
    L, U = np.zeros((n, n)), np.eye(n)
    for k in range(n):
        L[k:, k] = A[k:, k] - L[k:, :k] @ U[:k, k]                     # column k of L
        U[k, k+1:] = (A[k, k+1:] - L[k, :k] @ U[:k, k+1:]) / L[k, k]   # row k of U
    return L, U


Ld, Ud = doolittle(A)
Lc, Uc = crout(A)
print("Doolittle agrees with mylu:", np.allclose(Ld, L) and np.allclose(Ud, U))
print("\nCrout L =\n", Lc)
print("Crout U =\n", Uc)
print("\n||Lc Uc - A|| =", np.linalg.norm(Lc @ Uc - A))

The slides say that if $A = LU$ is Doolittle and $D = \operatorname{diag}(U)$, then $A = (LD)(D^{-1}U)$ is Crout. Check it:

In [ ]:
D = np.diag(np.diag(Ud))
print("Lc == Ld D      :", np.allclose(Lc, Ld @ D))
print("Uc == D^-1 Ud   :", np.allclose(Uc, np.linalg.solve(D, Ud)))

## 4. Factor once, solve many times

$A = LU$ turns each new right-hand side into two triangular solves:

$$Ax = b \;\Longrightarrow\; Lz = b \;\;(\text{forward}), \qquad Ux = z \;\;(\text{backward}).$$

In [ ]:
def lu_solve(L, U, b):
    return backward(U, forward(L, b))


b = np.array([1., 2., 3., 4.])
x = lu_solve(L, U, b)
print("x           =", x)
print("np.linalg   =", np.linalg.solve(A, b))

**The experiment the whole lecture is about.** We have $m$ right-hand sides that arrive *one at a time* (so we cannot stack them). Option A re-runs the elimination for each one; option B factors once and then does two triangular solves per right-hand side. We use SciPy's compiled routines so that the timings reflect the arithmetic, not Python loop overhead:

- `sla.solve(A, b)` — factor and solve, every time;
- `sla.lu_factor(A)` once, then `sla.lu_solve(lu, b)` for each $b$.

In [ ]:
n, m = 1000, 50          # <-- try other values
A_big = rng.standard_normal((n, n))
Bs = [rng.standard_normal(n) for _ in range(m)]

t0 = time.perf_counter()
X_a = [sla.solve(A_big, b) for b in Bs]
t_a = time.perf_counter() - t0

t0 = time.perf_counter()
lu = sla.lu_factor(A_big)
X_b = [sla.lu_solve(lu, b) for b in Bs]
t_b = time.perf_counter() - t0

print(f"re-eliminate every time : {t_a:.3f} s")
print(f"factor once, then solve : {t_b:.3f} s   (speed-up {t_a / t_b:.1f}x)")
print("same answers:", all(np.allclose(xa, xb) for xa, xb in zip(X_a, X_b)))

### ✏️ Your turn
Predict the speed-up from the operation counts: option A costs about $m \cdot 2n^3/3$ flops, option B about $2n^3/3 + m \cdot 2n^2$. Compute the predicted ratio for your $n$ and $m$ and compare. What happens as $m \to \infty$?

In [ ]:
flops_a = m * 2 * n**3 / 3
flops_b = 2 * n**3 / 3 + m * 2 * n**2
print(f"predicted speed-up = {flops_a / flops_b:.1f}x")

## 5. When does $A = LU$ exist? Leading principal minors

**Theorem.** If $\det(A_k) \ne 0$ for $k = 1, \dots, n$, then $A$ has a unique factorization $A = LU$ with $\ell_{ii} = 1$.

In fact the pivots are ratios of consecutive minors, $u_{kk} = \det(A_k)/\det(A_{k-1})$, so a zero minor is exactly a zero pivot.

In [ ]:
def leading_minors(A):
    return np.array([np.linalg.det(A[:k, :k]) for k in range(1, A.shape[0] + 1)])


dets = leading_minors(A)
print("det(A_k)                 :", dets)
print("det(A_k) / det(A_(k-1))  :", dets / np.concatenate([[1.0], dets[:-1]]))
print("diag(U)                  :", np.diag(U))

Now a nonsingular matrix whose second leading minor is zero. Watch `mylu` break down (NumPy prints a warning and returns `inf`/`nan` rather than raising).

In [ ]:
B = np.array([[1., 2., 3.],
              [2., 4., 5.],
              [1., 3., 4.]])
print("det(B)    =", np.linalg.det(B))
print("det(B_k)  =", leading_minors(B))
LB, UB = mylu(B)
print("U from mylu =\n", UB)

# Part II. Matrices that factor without a fight

## 6. Strictly diagonally dominant matrices

$|a_{ii}| > \sum_{j\ne i}|a_{ij}|$ for every row. The slides prove two things we can test:

1. elimination preserves dominance — every Schur complement $A^{(k)}$ is again SDD;
2. if $A^T$ is SDD (column dominance), partial pivoting never swaps rows.

In [ ]:
def is_sdd(A):
    d = np.abs(np.diag(A))
    off = np.sum(np.abs(A), axis=1) - d
    return bool(np.all(d > off))


def random_sdd(n, margin=0.1):
    A = rng.standard_normal((n, n))
    np.fill_diagonal(A, 0.0)
    r = np.sum(np.abs(A), axis=1)
    np.fill_diagonal(A, (r + margin) * rng.choice([-1, 1], n))
    return A


A_sdd = random_sdd(8)
print("A is SDD:", is_sdd(A_sdd))

# run elimination and test every trailing block
Ak = A_sdd.copy()
for k in range(A_sdd.shape[0] - 1):
    Ak[k+1:, k:] -= np.outer(Ak[k+1:, k] / Ak[k, k], Ak[k, k:])
    S = Ak[k+1:, k+1:]
    print(f"after step {k+1}: trailing {S.shape[0]}x{S.shape[0]} block SDD? {is_sdd(S)}")

## 7. Symmetric positive definite matrices: Cholesky

$A$ is SPD iff $A = LL^T$ with $L$ lower triangular and $\ell_{ii} > 0$:

$$\ell_{ii} = \sqrt{a_{ii} - \textstyle\sum_{k<i} \ell_{ik}^2}, \qquad \ell_{ji} = \Big(a_{ji} - \textstyle\sum_{k<i}\ell_{jk}\ell_{ik}\Big)\Big/\ell_{ii}, \quad j > i.$$

We add one thing to the slide's code: an explicit check, so a non-SPD matrix raises an error instead of silently producing `nan`.

In [ ]:
def cholesky(A):
    # Factor a symmetric positive definite A as A = L @ L.T.
    n = A.shape[0]
    L = np.zeros((n, n))
    for i in range(n):
        d = A[i, i] - L[i, :i] @ L[i, :i]
        if d <= 0:
            raise np.linalg.LinAlgError(f"not positive definite (pivot {i} = {d:.3g})")
        L[i, i] = np.sqrt(d)
        for j in range(i+1, n):
            L[j, i] = (A[j, i] - L[j, :i] @ L[i, :i]) / L[i, i]
    return L


A_spd = np.array([[ 4., -2.,   8.],
                  [-2.,  2.,   1.],
                  [ 8.,  1., 141.]])
Lc = cholesky(A_spd)
print("L =\n", Lc)
print("matches np.linalg.cholesky:", np.allclose(Lc, np.linalg.cholesky(A_spd)))
print("leading minors:", leading_minors(A_spd))

**Cholesky as a test for positive definiteness.** On large matrices, attempting the factorization is the practical test. Try a symmetric matrix that is *not* positive definite:

In [ ]:
A_indef = np.array([[1., 2.],
                    [2., 1.]])            # eigenvalues 3 and -1
print("eigenvalues:", np.linalg.eigvalsh(A_indef))
try:
    cholesky(A_indef)
except np.linalg.LinAlgError as err:
    print("cholesky failed:", err)

### ✏️ Your turn: $A = LDL^T$ without square roots
The slides note that square roots can be avoided by writing $A = LDL^T$ with $\ell_{ii} = 1$ and $D$ diagonal. Complete `ldlt` below. Derive the recurrence from $a_{ji} = \sum_{k \le i} \ell_{jk} d_k \ell_{ik}$:

$$d_i = a_{ii} - \sum_{k<i} \ell_{ik}^2 d_k, \qquad \ell_{ji} = \Big(a_{ji} - \sum_{k<i} \ell_{jk} d_k \ell_{ik}\Big)\Big/ d_i.$$

Then check that $L\sqrt{D}$ equals the Cholesky factor.

In [ ]:
def ldlt(A):
    # A = L D L^T with unit lower-triangular L and diagonal d (returned as a vector).
    n = A.shape[0]
    L, d = np.eye(n), np.zeros(n)
    for i in range(n):
        d[i] = A[i, i] - (L[i, :i]**2) @ d[:i]
        for j in range(i+1, n):
            L[j, i] = (A[j, i] - (L[j, :i] * d[:i]) @ L[i, :i]) / d[i]
    return L, d


L1, d = ldlt(A_spd)
print("d =", d)
print("L sqrt(D) == Cholesky factor:", np.allclose(L1 @ np.diag(np.sqrt(d)), Lc))

**Half the work of LU.** Cholesky costs about $n^3/3$ flops (i.e. $n^3/6$ multiply–add pairs) against $2n^3/3$ for LU. Compare the compiled LAPACK routines on the same SPD matrix:

In [ ]:
print(f"{'n':>6} {'LU (s)':>10} {'Cholesky (s)':>13} {'ratio':>7}")
for n in [500, 1000, 2000]:          # <-- push higher if your machine allows
    G = rng.standard_normal((n, n))
    S = G @ G.T + n * np.eye(n)       # SPD
    t_lu = time_it(sla.lu_factor, S)
    t_ch = time_it(sla.cho_factor, S)
    print(f"{n:>6} {t_lu:>10.4f} {t_ch:>13.4f} {t_lu / t_ch:>7.2f}")

## 8. Tridiagonal matrices: the whole factorization in $O(n)$

Store only the three diagonals: `a` (sub, length $n-1$), `b` (main, length $n$), `c` (super, length $n-1$). $L$ has one sub-diagonal `l`, and $U$ has diagonal `u` and super-diagonal `c` unchanged.

In [ ]:
def tridiag_lu(a, b, c):
    # L and U for a tridiagonal matrix given by its diagonals.
    n = len(b)
    l, u = np.zeros(n-1), np.zeros(n)
    u[0] = b[0]
    for i in range(n-1):
        l[i] = a[i] / u[i]
        u[i+1] = b[i+1] - l[i] * c[i]
    return l, u


def tridiag_solve(l, u, c, f):
    # Solve (L U) x = f using the tridiagonal factors: forward, then backward.
    n = len(u)
    z = np.zeros(n)
    z[0] = f[0]
    for i in range(1, n):
        z[i] = f[i] - l[i-1] * z[i-1]
    x = np.zeros(n)
    x[-1] = z[-1] / u[-1]
    for i in range(n-2, -1, -1):
        x[i] = (z[i] - c[i] * x[i+1]) / u[i]
    return x

**Application: a two-point boundary value problem.** Solve $-y'' = \pi^2 \sin(\pi t)$ on $(0, 1)$ with $y(0) = y(1) = 0$; the exact solution is $y = \sin(\pi t)$. The centered difference $-(y_{i-1} - 2y_i + y_{i+1})/h^2$ gives a tridiagonal system with $2$ on the diagonal and $-1$ off it (and it is SPD).

In [ ]:
N = 200                                  # interior points
h = 1.0 / (N + 1)
t = np.linspace(h, 1 - h, N)
a = -np.ones(N-1); b = 2 * np.ones(N); c = -np.ones(N-1)
f = h**2 * np.pi**2 * np.sin(np.pi * t)

l, u = tridiag_lu(a, b, c)
y = tridiag_solve(l, u, c, f)

T = np.diag(b) + np.diag(a, -1) + np.diag(c, 1)
print("matches dense solve :", np.allclose(y, np.linalg.solve(T, f)))
print("max error vs sin(pi t):", np.max(np.abs(y - np.sin(np.pi * t))))

plt.figure(figsize=(6, 3.5))
plt.plot(t, np.sin(np.pi * t), "k-", label="exact")
plt.plot(t[::10], y[::10], "o", label="tridiagonal solve")
plt.xlabel("t"); plt.ylabel("y"); plt.legend(); plt.title("-y'' = pi^2 sin(pi t)")
plt.show()

**Timing: $O(n)$ versus $O(n^3)$.** Our tridiagonal solver is a plain Python loop, and the dense solver is compiled LAPACK. Even so, the structure wins once $n$ is moderate. Look at how each time grows when $n$ doubles.

In [ ]:
print(f"{'n':>6} {'tridiagonal (s)':>16} {'dense LAPACK (s)':>17}")
for N in [500, 1000, 2000, 4000]:
    a = -np.ones(N-1); b = 2 * np.ones(N); c = -np.ones(N-1); f = rng.standard_normal(N)
    t_tri = time_it(lambda: tridiag_solve(*tridiag_lu(a, b, c), c, f))
    T = np.diag(b) + np.diag(a, -1) + np.diag(c, 1)
    t_den = time_it(np.linalg.solve, T, f)
    print(f"{N:>6} {t_tri:>16.4f} {t_den:>17.4f}")

### ✏️ Your turn
1. For $n = 10^6$ the dense matrix would need $8n^2 = 8$ TB of memory. Run `tridiag_lu` + `tridiag_solve` at $n = 10^6$ (it takes a second or two) and check the max error against $\sin(\pi t)$.
2. SciPy's `sla.solve_banded` does the same job in compiled code. Look up its `ab` storage format and time it against our loop at $n = 10^6$.

In [ ]:
# your code here

# Part III. Row interchanges: $PA = LU$

## 9. A good matrix with no LU factorization

In [ ]:
A2 = np.array([[0., 2.],
               [3., 1.]])
print("det(A) =", np.linalg.det(A2), "  but det(A_1) =", A2[0, 0])
L2, U2 = mylu(A2)
print("mylu gives U =\n", U2)

## 10. LU with partial pivoting

At step $i$, find the row $p \ge i$ with the largest $|a_{pi}|$, swap rows $i$ and $p$ of the working matrix **and of the multipliers already stored in $L$**, then eliminate. The result satisfies $PA = LU$, with $\det A = (-1)^{\text{swaps}} \prod u_{ii}$.

In [ ]:
def lupp(A):
    # Factor P @ A = L @ U using partial pivoting.
    A = A.astype(float).copy()
    n = A.shape[0]
    P, L, swaps = np.eye(n), np.zeros((n, n)), 0
    for i in range(n-1):
        p = i + np.argmax(np.abs(A[i:, i]))
        if p != i:
            swaps += 1
            A[[i, p]], P[[i, p]] = A[[p, i]], P[[p, i]]
            L[[i, p], :i] = L[[p, i], :i]      # move the multipliers too
        L[i+1:, i] = A[i+1:, i] / A[i, i]
        A[i+1:, i:] -= np.outer(L[i+1:, i], A[i, i:])
    return L + np.eye(n), np.triu(A), P, swaps

**The $3\times 3$ example from the slides.** Expected: two interchanges, $U$ with diagonal $2, 2, -9/4$, $\det A = -9$, and for $b = (5, 0, 2)^T$ the solution $x = (-1, 1, 1)^T$.

In [ ]:
A3 = np.array([[0., 2., 3.],
               [1., 1., 0.],
               [2., 1., 3.]])
L3, U3, P3, swaps = lupp(A3)
print("P =\n", P3); print("L =\n", L3); print("U =\n", U3)
print("PA == LU :", np.allclose(P3 @ A3, L3 @ U3))
print("det(A) =", (-1)**swaps * np.prod(np.diag(U3)), "   numpy:", np.linalg.det(A3))

b3 = np.array([5., 0., 2.])
z = forward(L3, P3 @ b3)
x = backward(U3, z)
print("\nPb =", P3 @ b3, "\nz  =", z, "\nx  =", x)

**Comparing with SciPy.** `scipy.linalg.lu` returns `P, L, U` with $A = PLU$, the *transpose* convention. So SciPy's `P` is our `P.T`. Always check a library's convention before you trust its `P`.

In [ ]:
Ps, Ls, Us = sla.lu(A3)
print("A == Ps Ls Us :", np.allclose(A3, Ps @ Ls @ Us))
print("Ps == P.T     :", np.allclose(Ps, P3.T))
print("same L, U     :", np.allclose(Ls, L3) and np.allclose(Us, U3))

## 11. The line everybody forgets

Below is `lupp` with the swap of the stored multipliers removed. On matrices where the pivot row changes after step 1, the factors no longer satisfy $PA = LU$.

In [ ]:
def lupp_buggy(A):
    A = A.astype(float).copy()
    n = A.shape[0]
    P, L = np.eye(n), np.zeros((n, n))
    for i in range(n-1):
        p = i + np.argmax(np.abs(A[i:, i]))
        if p != i:
            A[[i, p]], P[[i, p]] = A[[p, i]], P[[p, i]]
            # (forgot to swap L)
        L[i+1:, i] = A[i+1:, i] / A[i, i]
        A[i+1:, i:] -= np.outer(L[i+1:, i], A[i, i:])
    return L + np.eye(n), np.triu(A), P


Lb, Ub, Pb = lupp_buggy(A3)
print("buggy: ||PA - LU|| =", np.linalg.norm(Pb @ A3 - Lb @ Ub))

M6 = rng.standard_normal((6, 6))
for f in (lupp, lupp_buggy):
    Lf, Uf, Pf = f(M6)[:3]
    print(f"{f.__name__:>11} on random 6x6: ||PA - LU|| = {np.linalg.norm(Pf @ M6 - Lf @ Uf):.2e}")

### ✏️ Your turn (the "to prepare" task from the slides)
1. Run `lupp` on a random $6\times 6$ matrix and compare its `P`, `L`, `U` with `sla.lu`. Remember the transpose.
2. Check that every entry of $L$ from `lupp` satisfies $|\ell_{ij}| \le 1$. Why must this be true?
3. Generate a random matrix whose *transpose* is SDD (`random_sdd(8).T`) and confirm that `lupp` reports `swaps == 0`.

In [ ]:
# your code here

# Part IV. Conditioning and stability

## 12. Subordinate norms and the condition number

$\|A\|_\infty = \sup_{x\ne 0} \|Ax\|_\infty/\|x\|_\infty$ is the maximum absolute **row sum**. We check the formula against random sampling of the sup (which approaches it from below), and against the maximizing vector $x = \operatorname{sign}(\text{row}_i)$.

In [ ]:
A = rng.standard_normal((5, 5))
row_sum = np.max(np.sum(np.abs(A), axis=1))

X = rng.uniform(-1, 1, (5, 20000))
sampled = np.max(np.max(np.abs(A @ X), axis=0) / np.max(np.abs(X), axis=0))

i = np.argmax(np.sum(np.abs(A), axis=1))
x_star = np.sign(A[i])
attained = np.max(np.abs(A @ x_star)) / np.max(np.abs(x_star))

print(f"max row sum          = {row_sum:.6f}")
print(f"np.linalg.norm(inf)  = {np.linalg.norm(A, np.inf):.6f}")
print(f"best of 20000 samples= {sampled:.6f}")
print(f"x = sign(row i)      = {attained:.6f}")

**The ill-conditioned example from the slides.** Expected: $\|A\|_\infty \approx 20$, $\|A^{-1}\|_\infty \approx 1.34\times10^6$, $\kappa_\infty(A) \approx 2.68\times10^7$.

In [ ]:
A = np.array([[ 1e-6,   1.,  1.],
              [-1e-10, 15., -5.],
              [ 0.,    11.,  2.]])
nA = np.linalg.norm(A, np.inf)
nAi = np.linalg.norm(np.linalg.inv(A), np.inf)
print(f"||A||_inf      = {nA:.10g}")
print(f"||A^-1||_inf   = {nAi:.4e}")
print(f"kappa_inf(A)   = {nA * nAi:.4e}   np.linalg.cond: {np.linalg.cond(A, np.inf):.4e}")

## 13. Testing the perturbation bound

For $Ax = b$ and $A\tilde x = b + \tilde b$:
$$\frac{\|x - \tilde x\|}{\|x\|} \le \kappa(A)\,\frac{\|\tilde b\|}{\|b\|}.$$

We perturb $b$ randomly many times and plot the **amplification** (relative error in $x$) / (relative error in $b$). The bound says it never exceeds $\kappa(A)$.

In [ ]:
def amplification(A, trials=2000, size=1e-10):
    n = A.shape[0]
    x = np.ones(n)
    b = A @ x
    out = np.empty(trials)
    for k in range(trials):
        db = size * rng.standard_normal(n)
        xt = np.linalg.solve(A, b + db)
        out[k] = (np.linalg.norm(xt - x, np.inf) / np.linalg.norm(x, np.inf)) / \
                 (np.linalg.norm(db, np.inf) / np.linalg.norm(b, np.inf))
    return out


amp = amplification(A)
kap = np.linalg.cond(A, np.inf)
plt.figure(figsize=(6, 3.5))
plt.hist(np.log10(amp), bins=40)
plt.axvline(np.log10(kap), color="k", ls="--", label=f"log10 kappa = {np.log10(kap):.2f}")
plt.xlabel("log10(amplification)"); plt.ylabel("count"); plt.legend()
plt.title("Error amplification under random perturbations of b")
plt.show()
print(f"largest observed amplification = {amp.max():.3e}  <=  kappa = {kap:.3e}")

Random perturbations rarely hit the worst case. The bound is attained when $\tilde b$ points along the direction that $A^{-1}$ stretches the most **and** $b$ is the direction $A$ stretches the most.

### ✏️ Your turn
Using the SVD $A = U\Sigma V^T$ (with the 2-norm), choose $b = u_1$ (the top left singular vector) and $\tilde b = \delta\, u_n$ (the bottom one). Show numerically that the amplification then equals $\kappa_2(A) = \sigma_1/\sigma_n$ exactly.

In [ ]:
Us, s, Vt = np.linalg.svd(A)
b = Us[:, 0]
db = 1e-10 * Us[:, -1]
x = np.linalg.solve(A, b)
xt = np.linalg.solve(A, b + db)
amp2 = (np.linalg.norm(xt - x) / np.linalg.norm(x)) / (np.linalg.norm(db) / np.linalg.norm(b))
print(f"amplification = {amp2:.6e}")
print(f"kappa_2(A)    = {s[0] / s[-1]:.6e}")

## 14. A two-by-two warning

Changing $b$ in its thirteenth digit moves the solution from $(1, 1)$ to $(2, -1)$.

In [ ]:
A = np.array([[2 + 1e-13, 1.0],
              [2.0,       1.0]])
b = np.array([3 + 1e-13, 3.0])
print("solve(A, b)          =", np.linalg.solve(A, b))
print("solve(A, b + db)     =", np.linalg.solve(A, b + np.array([1e-13, 0.0])))
print(f"cond_inf(A)          = {np.linalg.cond(A, np.inf):.4e}")

## 15. A small residual proves nothing

$$\frac{\|x - \tilde x\|}{\|x\|} \le \kappa(A)\frac{\|r\|}{\|b\|}, \qquad r = b - A\tilde x.$$

The Hilbert matrix $H_{ij} = 1/(i + j - 1)$ is the classic ill-conditioned matrix. For each $n$ we solve $Hx = b$ with exact solution $x = (1, \dots, 1)$, and compare the relative residual with the relative error.

In [ ]:
print(f"{'n':>3} {'cond(H)':>10} {'rel residual':>13} {'rel error':>10} {'kappa*resid':>12}")
for n in range(2, 15, 2):
    H = sla.hilbert(n)
    x_true = np.ones(n)
    bH = H @ x_true
    xh = np.linalg.solve(H, bH)
    k = np.linalg.cond(H, np.inf)
    rr = np.linalg.norm(bH - H @ xh, np.inf) / np.linalg.norm(bH, np.inf)
    re = np.linalg.norm(xh - x_true, np.inf) / np.linalg.norm(x_true, np.inf)
    print(f"{n:>3} {k:>10.2e} {rr:>13.2e} {re:>10.2e} {k * rr:>12.2e}")

**Observe.** The residual stays near machine precision for every $n$, while the error grows with $\kappa(H)$. At $n = 12$ or so, no digit of $x$ is correct, yet $A\tilde x \approx b$ looks perfect.

## 16. Iterative refinement

1. factor $PA = LU$ (in low precision, $t$ digits);
2. solve $LUx_0 = Pb$;
3. $r_k = b - Ax_k$ in **higher** precision;
4. solve $LU\,x_c = Pr_k$, update $x_{k+1} = x_k + x_c$; repeat.

### 16a. The slide example, in three-digit arithmetic
We simulate a three-significant-digit machine by rounding after every operation with `fl`. The residual is computed in double. Expected: $x^{(0)} = (2.11, -3.17)$, $x^{(1)} = (1.99, -2.99)$, $x^{(2)} = (2.00, -3.00)$.

In [ ]:
def fl(v):
    # Round to 3 significant decimal digits.
    return float(f"{v:.3g}")


A = np.array([[0.986, 0.579],
              [0.409, 0.237]])
b = np.array([0.235, 0.107])

# LU in 3-digit arithmetic (0.986 is already the largest pivot, so P = I)
l21 = fl(A[1, 0] / A[0, 0])
u22 = fl(A[1, 1] - fl(l21 * A[0, 1]))
print(f"l21 = {l21},  u22 = {u22}   (u22 has lost almost every digit)")


def solve3(r):
    # Forward and backward substitution in 3-digit arithmetic.
    z1 = fl(r[1] - fl(l21 * r[0]))
    x1 = fl(z1 / u22)
    x0 = fl(fl(r[0] - fl(A[0, 1] * x1)) / A[0, 0])
    return np.array([x0, x1])


x = solve3(b)
print(f"x^(0) = {x}")
for k in range(1, 4):
    r = b - A @ x                      # residual in double precision
    x = np.array([fl(v) for v in x + solve3(r)])
    print(f"x^({k}) = {x}")
print(f"kappa_inf(A) = {np.linalg.cond(A, np.inf):.0f}")

### 16b. Mixed precision: factor in `float32`, refine in `float64`
This is how refinement is used today: factor cheaply in single precision ($d \approx 7$ digits), compute residuals in double. With $\kappa \approx 10^q$, the slides predict about $\min(16, (k+1)(7 - q))$ correct digits after $k$ steps.

In [ ]:
def refine_mixed(A, b, steps=6):
    lu32 = sla.lu_factor(A.astype(np.float32))                     # t-digit factorization
    x = sla.lu_solve(lu32, b.astype(np.float32)).astype(np.float64)
    history = [x.copy()]
    for _ in range(steps):
        r = b - A @ x                                              # double-precision residual
        xc = sla.lu_solve(lu32, r.astype(np.float32))
        x = x + xc.astype(np.float64)
        history.append(x.copy())
    return history


n, q = 200, 3                               # <-- try q = 1, 3, 5, 7
Uq, _ = np.linalg.qr(rng.standard_normal((n, n)))
Vq, _ = np.linalg.qr(rng.standard_normal((n, n)))
A = Uq @ np.diag(np.logspace(0, -q, n)) @ Vq.T      # kappa_2(A) = 10^q
x_true = rng.standard_normal(n)
b = A @ x_true

print(f"kappa_2(A) = {np.linalg.cond(A):.1e}")
print(f"{'k':>3} {'rel error':>11} {'correct digits':>15} {'predicted':>10}")
for k, xk in enumerate(refine_mixed(A, b)):
    e = rel_err(xk, x_true)
    print(f"{k:>3} {e:>11.2e} {-np.log10(e):>15.1f} {min(16, (k+1)*(7-q)):>10}")

**Observe.** Each step adds roughly $7 - q$ digits, just as predicted. The error then levels off near $\kappa(A) \cdot 10^{-16} \approx 10^{-13}$, not at $10^{-16}$. The final answer can be no more accurate than the problem allows, even with a double-precision residual. Refinement recovers the digits lost to the *low-precision factorization*, not the ones lost to conditioning.

### ✏️ Your turn
Set `q = 7` or larger. The factorization in `float32` is then barely (or not) better than nothing, so $\|I - B^{-1}A\| < 1$ fails. What happens to the iterates? Relate this to the convergence theorem on the slides ($\|A^{-1}(B - A)\| < 1/2$).

## 17. A well-conditioned matrix that elimination gets wrong

$$A = \begin{bmatrix}\varepsilon & 1\\ 1 & 1\end{bmatrix}, \qquad \kappa_\infty(A) = \frac{4}{1-\varepsilon} \approx 4.$$

The problem is fine. Without pivoting, $u_{22} = 1 - 1/\varepsilon$ rounds to $-1/\varepsilon$ and the $1$ is lost. We measure the **backward error** $\|LU - PA\|/\|A\|$ of the computed factors, and the forward error of the solution with $b$ chosen so that $x \approx (1, 1)$.

In [ ]:
print(f"{'eps':>8} {'cond':>6} {'backward err (no piv)':>22} {'backward err (piv)':>19} "
      f"{'x (no piv)':>24} {'x (piv)':>22}")
for k in [4, 8, 12, 16, 20]:
    eps = 10.0 ** (-k)
    A = np.array([[eps, 1.], [1., 1.]])
    b = np.array([1. + eps, 2.])                  # exact x = (1, 1)
    Ln, Un = mylu(A)
    Lp, Up, Pp, _ = lupp(A)
    be_n = np.linalg.norm(Ln @ Un - A, np.inf) / np.linalg.norm(A, np.inf)
    be_p = np.linalg.norm(Lp @ Up - Pp @ A, np.inf) / np.linalg.norm(A, np.inf)
    xn = lu_solve(Ln, Un, b)
    xp = lu_solve(Lp, Up, Pp @ b)
    print(f"{eps:>8.0e} {np.linalg.cond(A, np.inf):>6.2f} {be_n:>22.2e} {be_p:>19.2e} "
          f"{str(np.round(xn, 8)):>24} {str(np.round(xp, 8)):>22}")

**Read the table.** $\kappa \approx 4$ in every row, so the *problem* never changes. Without pivoting the backward error grows to $O(1)$: the factors belong to a matrix nowhere near $A$ (unstable). With pivoting it stays at machine precision (backward stable), and the solution is right.

## 18. The growth factor, and Wilkinson's worst case

$$g_n(A) = \frac{\max_{i,j,k} |a^{(k)}_{ij}|}{\max_{i,j} |a_{ij}|} \le 2^{n-1} \quad \text{(partial pivoting)}.$$

The bound is attained by the matrix with $1$ on the diagonal and in the last column, and $-1$ below the diagonal. We instrument `lupp` to record $g_n$.

In [ ]:
def growth_factor(A):
    # Growth factor of Gaussian elimination with partial pivoting.
    A = A.astype(float).copy()
    n = A.shape[0]
    a0 = np.max(np.abs(A))
    gmax = a0
    for i in range(n-1):
        p = i + np.argmax(np.abs(A[i:, i]))
        A[[i, p]] = A[[p, i]]
        A[i+1:, i:] -= np.outer(A[i+1:, i] / A[i, i], A[i, i:])
        gmax = max(gmax, np.max(np.abs(A[i+1:, i+1:])))
    return gmax / a0


def wilkinson(n):
    W = np.eye(n) - np.tril(np.ones((n, n)), -1)
    W[:, -1] = 1.0
    return W


print(wilkinson(5))
print(f"\n{'n':>4} {'g_n(Wilkinson)':>15} {'2^(n-1)':>10} {'g_n(random)':>12}")
for n in [5, 10, 20, 40, 60]:
    print(f"{n:>4} {growth_factor(wilkinson(n)):>15.3e} {2.0**(n-1):>10.3e} "
          f"{growth_factor(rng.standard_normal((n, n))):>12.2f}")

Wilkinson's matrix is well conditioned, yet with partial pivoting the solution loses every digit around $n \approx 60$, just as the bound $4n^2 g_n \kappa\, \mathrm{Eps}$ allows.

In [ ]:
print(f"{'n':>4} {'cond(W)':>9} {'rel error, PA=LU':>18} {'rel error, QR':>15}")
for n in [20, 40, 50, 55, 60, 64]:
    W = wilkinson(n)
    x_true = rng.standard_normal(n)
    bW = W @ x_true
    L_, U_, P_, _ = lupp(W)
    x_lu = lu_solve(L_, U_, P_ @ bW)
    Q, R = np.linalg.qr(W)
    x_qr = sla.solve_triangular(R, Q.T @ bW)
    print(f"{n:>4} {np.linalg.cond(W):>9.1f} {rel_err(x_lu, x_true):>18.2e} {rel_err(x_qr, x_true):>15.2e}")

The QR column is a preview: an orthogonal factorization has no growth factor to worry about. In practice, though, such growth essentially never happens for matrices that arise in applications. The random column above stays small, which is why partial pivoting remains the default in every library.

# Exercises

1. **Operation counts.** Instrument `mylu` to count multiplications and divisions, and verify that the count is $\approx n^3/3$ for large $n$. Do the same for `cholesky` and confirm $\approx n^3/6$.
2. **Banded LU.** Generalize `tridiag_lu` to a matrix with lower bandwidth $n_\ell$ and upper bandwidth $n_u$, stored densely but with loop bounds $\min(i + n_\ell, n)$ and $\min(i + n_u, n)$. Check that $L$ and $U$ stay inside the band, and measure how the time scales with $n$ for fixed bandwidth.
3. **Pivoting destroys the band.** Take a random tridiagonal matrix that is *not* diagonally dominant and factor it with `lupp`. How wide does $U$ become? Explain.
4. **Determinant by LU.** Write `det_lu(A)` using `lupp`. Compare with `np.linalg.det` on random $n\times n$ matrices, and explain why for $n = 1000$ you should compute $\log|\det A| = \sum \log|u_{ii}|$ instead.
5. **Cholesky versus LU on SPD matrices.** For SPD $A$, compare the $U$ from `mylu` with $L^T$ from `cholesky`. Show that $U = D L_1^T$, where $A = L_1 D L_1^T$ from `ldlt`.
6. **Condition estimation.** Computing $\|A^{-1}\|$ exactly costs as much as the inverse. Using the factors from `lupp`, estimate $\|A^{-1}\|_\infty$ by solving $Ay = d$ for a few random $d$ with entries $\pm 1$, and taking $\max \|y\|_\infty$. How close does it get to the true value on the Hilbert matrix? (This is the idea behind LAPACK's `gecon`.)
7. **Refinement with a bad factorization.** Replace the `float32` factorization in `refine_mixed` by the factors of $A + E$ for a random perturbation $E$ with $\|E\|/\|A\| = 10^{-s}$. Find the value of $s$ below which the iteration stops converging, and compare with the theorem's condition $\|A^{-1}E\| < 1/2$.